In [0]:
import os
import sys

dbutils.widgets.text("catalog", "lakemeter_catalog")
dbutils.widgets.text("schema", "ecommerce_dev")
dbutils.widgets.text("volume", "raw_data")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
raw_path = f"/Volumes/{catalog}/{schema}/{volume}"

sys.dont_write_bytecode = True
lib_path = os.path.abspath(os.path.join(os.getcwd(), "..", "pipeline"))
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

print(f"Writing synthetic data to {raw_path}")

In [0]:
from ecommerce_lib.synthetic import generate_dataframes

raw = generate_dataframes(spark)

# Overwrite on every run so CI always tests the same deterministic dataset.
raw["customers"].write.mode("overwrite").option("header", "true").csv(f"{raw_path}/customers")
raw["products"].write.mode("overwrite").json(f"{raw_path}/products")
raw["orders"].write.mode("overwrite").parquet(f"{raw_path}/orders")

In [0]:
from ecommerce_lib.schemas import RAW_SCHEMAS
from ecommerce_lib.synthetic import EXPECTED_COUNTS

readers = {
    "customers": lambda p: spark.read.option("header", "true").schema(RAW_SCHEMAS["customers"]).csv(p),
    "products": lambda p: spark.read.schema(RAW_SCHEMAS["products"]).json(p),
    "orders": lambda p: spark.read.schema(RAW_SCHEMAS["orders"]).parquet(p),
}
for entity, read in readers.items():
    n = read(f"{raw_path}/{entity}").count()
    expected = EXPECTED_COUNTS[f"bronze_{entity}"]
    assert n == expected, f"{entity}: wrote {n} rows, expected {expected}"
    print(f"{entity}: {n} rows")